# AFL Data Cleaning Notebook

In [3]:
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

raw = pd.read_csv('../../afl_players_round_by_round_stats_raw.csv')
clean = pd.read_csv('../../round_by_round_cleaned.csv')
tm = pd.read_csv('../../team_matches_home_away_raw.csv')

print('raw shape:', raw.shape)
print('clean shape:', clean.shape)
print('team_matches shape:', tm.shape)


raw shape: (274089, 36)
clean shape: (273082, 37)
team_matches shape: (15808, 19)


## 1. Compare raw vs pre-cleaned player stats

Check how many rows differ and why.

In [4]:
raw_ids = set(raw['id'])
clean_ids = set(clean['id'])

print('raw ids:', len(raw_ids))
print('clean ids:', len(clean_ids))
print('in raw but not in clean:', len(raw_ids - clean_ids))
print('in clean but not in raw:', len(clean_ids - raw_ids))


raw ids: 274079
clean ids: 273082
in raw but not in clean: 997
in clean but not in raw: 0


In [5]:
# exact duplicate rows in raw
print('exact duplicate rows in raw:', raw.duplicated().sum())
dups = raw[raw.duplicated(keep=False)].sort_values('id')
dups.head(6)


exact duplicate rows in raw: 10


,id,team,year,career_game_count,opponent,round,result,jersey_num,kicks,marks,handballs,disposals,goals,behinds,hit_outs,tackles,rebound_50s,inside_50s,clearances,clangers,free_kicks_for,free_kicks_against,brownlow_votes,contested_possessions,uncontested_possessions,contested_marks,marks_inside_50,one_percenters,bounces,goal_assist,percentage_of_game_played,player_id,match_date,fantasy_points,score,margin
9,501586,Port Adelaide Power,1997,69,Brisbane Lions,20,D,39,12.0,7.0,6.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45591,1997-08-16,68,NaN,0
20972,501586,Port Adelaide Power,1997,69,Brisbane Lions,20,D,39,12.0,7.0,6.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45591,1997-08-16,68,NaN,0
8,507061,Fremantle Dockers,1997,37,St Kilda Saints,20,L,38,4.0,2.0,8.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45630,1997-08-16,36,NaN,-13
20971,507061,Fremantle Dockers,1997,37,St Kilda Saints,20,L,38,4.0,2.0,8.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45630,1997-08-16,36,NaN,-13
20969,507101,Fremantle Dockers,2002,77,St Kilda Saints,2,W,5,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,45630,2002-04-07,3,NaN,3
6,507101,Fremantle Dockers,2002,77,St Kilda Saints,2,W,5,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,45630,2002-04-07,3,NaN,3


In [6]:
# rows with null or negative disposals (physically impossible to have negative disposals)
bad_disposals = raw[(raw['disposals'].isna()) | (raw['disposals'] < 0)]
print('rows with null disposals:', raw['disposals'].isna().sum())
print('rows with negative disposals:', (raw['disposals'] < 0).sum())
print('total bad disposal rows:', len(bad_disposals))
bad_disposals[['id','team','opponent','match_date','disposals','kicks','handballs']].head()


rows with null disposals: 8453
rows with negative disposals: 723
total bad disposal rows: 9176


,id,team,opponent,match_date,disposals,kicks,handballs
1,614897,Geelong Cats,St Kilda Saints,2024-03-16,NaN,5.0,NaN
5,540025,St Kilda Saints,Fremantle Dockers,2002-04-07,NaN,NaN,NaN
6,507101,Fremantle Dockers,St Kilda Saints,2002-04-07,NaN,1.0,NaN
7,554875,Brisbane Lions,Adelaide Crows,2002-06-01,-3.0,1.0,1.0
13,521355,Fremantle Dockers,Sydney Swans,1999-08-21,NaN,1.0,NaN


In [7]:
# rows belonging to round '0' (the 2024/2025 AFL Opening Round)
round0 = raw[raw['round'] == '0']
print('round 0 rows in raw:', len(round0))
print('round 0 rows in clean:', (clean['round'] == '0').sum())
round0[['team','opponent','match_date','round','disposals','fantasy_points']].head()


round 0 rows in raw: 276
round 0 rows in clean: 0


,team,opponent,match_date,round,disposals,fantasy_points
3762,Sydney Swans,Melbourne Demons,2024-03-07,0,15.0,69
5462,Collingwood Magpies,Greater Western Sydney Giants,2024-03-09,0,13.0,75
5479,Collingwood Magpies,Greater Western Sydney Giants,2025-03-09,0,15.0,77
7045,Melbourne Demons,Sydney Swans,2024-03-07,0,7.0,48
7438,Hawthorn Hawks,Sydney Swans,2025-03-07,0,4.0,44


In [8]:
# confirm these three issues explain (almost) all of the removed rows
missing_ids = raw_ids - clean_ids
missing_rows = raw[raw['id'].isin(missing_ids)]

explained = set(raw[raw.duplicated()]['id']) | set(bad_disposals['id']) | set(round0['id'])
unexplained = missing_rows[~missing_rows['id'].isin(explained)]

print('rows removed between raw and clean:', len(missing_rows))
print('rows explained by (duplicate / bad disposals / round 0):', len(missing_rows) - len(unexplained))
print('rows still unexplained:', len(unexplained))


rows removed between raw and clean: 998
rows explained by (duplicate / bad disposals / round 0): 998
rows still unexplained: 0


**Finding:** the "cleaned" file removed rows for three reasons:

1. 10 exact duplicate rows
2. 9,176 rows with null or negative `disposals` (negative disposals is a genuine data bug)
3. 276 rows from round "0" (Opening Round, 2024/2025) - these look like they were dropped
   by accident, since they have normal stat lines and match real games in the team-match file

Decision: for the merge notebook, we will build the enriched dataset from the **raw** file
(after removing the 10 exact duplicates) so we don't lose the 276 valid Opening Round
records. Rows with bad `disposals` values are flagged rather than silently dropped, since
dropping the whole row throws away every other stat that player recorded that game.


## 2. Check the `score` column

In [9]:
print('score non-null in raw:', raw['score'].notna().sum(), '/', len(raw))
print('score non-null in clean:', clean['score'].notna().sum(), '/', len(clean))


score non-null in raw: 0 / 274089
score non-null in clean: 0 / 273082


**Finding:** `score` is 100% null in both the raw and the pre-cleaned files. This is a
dead column at the source, not something introduced during cleaning. It should be dropped
or ignored in downstream analysis.


## 3. Other quality checks on the player stats file

In [10]:
print('percentage_of_game_played missing:', raw['percentage_of_game_played'].isna().sum(), '/', len(raw))
print('negative fantasy_points rows:', (raw['fantasy_points'] < 0).sum())
print('fantasy_points range:', raw['fantasy_points'].min(), 'to', raw['fantasy_points'].max())
print('jersey_num range:', raw['jersey_num'].min(), 'to', raw['jersey_num'].max())
print('year range:', raw['year'].min(), 'to', raw['year'].max())


percentage_of_game_played missing: 69682 / 274089
negative fantasy_points rows: 167
fantasy_points range: -10 to 210
jersey_num range: 1 to 67
year range: 1983 to 2025


## 4. Inspect and clean `team_matches_home_away_raw.csv`

This file has several formatting issues that will break a merge if left as-is.

In [20]:
print('team_name unique (raw):')
for v in sorted(tm['team_name'].unique(), key=str.lower):
    print(v)


team_name unique (raw):
	 Adelaide Crows 
	Brisbane Bears
	Brisbane Lions
	Carlton Blues
	Collingwood Magpies
	Essendon Bombers
	Fitzroy Lions
	Fremantle Dockers
	Geelong Cats
	Gold Coast Suns
	Greater Western Sydney Giants
	Hawthorn Hawks
	Melbourne Demons
	North Melbourne Kangaroos
	Port Adelaide Power
	Richmond Tigers
	St Kilda Saints
	Sydney Swans
	W. Bulldogs
	West Coast Eagles
 Adelaide Crows 
Brisbane Bears
Brisbane Lions
Carlton Blues
Collingwood Magpies
Essendon Bombers
Fitzroy Lions
Fremantle Dockers
Geelong Cats
Gold Coast Suns
Greater Western Sydney Giants
Hawthorn Hawks
Melbourne Demons
North Melbourne Kangaroos
Port Adelaide Power
Richmond Tigers
St Kilda Saints
Sydney Swans
W. Bulldogs
West Coast Eagles


**Issues found in team_name / opponent:**
- Leading/trailing whitespace and tab characters, e.g. "\tHawthorn Hawks"
- "W. Bulldogs" used instead of "Western Bulldogs" in some rows
- Case inconsistency in opponent only (not team_name): e.g. "adelaide crows" instead
  of "Adelaide Crows", affecting 2,345 rows


In [12]:
# check case inconsistency in opponent specifically
lower_map = {}
for v in tm['opponent'].unique():
    lower_map.setdefault(v.strip().lower(), []).append(v)

case_issues = {k: v for k, v in lower_map.items() if len(v) > 1}
case_issues


{'north melbourne kangaroos': ['North Melbourne Kangaroos',
  'north melbourne kangaroos'],
 'hawthorn hawks': ['Hawthorn Hawks', 'hawthorn hawks'],
 'brisbane lions': ['Brisbane Lions', 'brisbane lions'],
 'melbourne demons': ['Melbourne Demons', 'melbourne demons'],
 'geelong cats': ['Geelong Cats', 'geelong cats'],
 'west coast eagles': ['West Coast Eagles', 'west coast eagles'],
 'st kilda saints': ['St Kilda Saints', 'st kilda saints'],
 'sydney swans': ['Sydney Swans', 'sydney swans'],
 'fremantle dockers': ['Fremantle Dockers', 'fremantle dockers'],
 'richmond tigers': ['Richmond Tigers', 'richmond tigers'],
 'port adelaide power': ['Port Adelaide Power', 'port adelaide power'],
 'carlton blues': ['Carlton Blues', 'carlton blues'],
 'collingwood magpies': ['Collingwood Magpies', 'collingwood magpies'],
 'essendon bombers': ['Essendon Bombers', 'essendon bombers'],
 'adelaide crows': ['Adelaide Crows', 'adelaide crows'],
 'fitzroy lions': ['Fitzroy Lions', 'fitzroy lions'],
 'wes

In [13]:
print('rows with lowercase opponent names:', tm['opponent'].str.islower().sum())


rows with lowercase opponent names: 2345


In [14]:
# check venue column for hidden characters
print('venue unique count before cleaning:', tm['venue'].nunique())
for v in sorted(tm['venue'].unique(), key=lambda x: x.lower())[:10]:
    print(repr(v))


venue unique count before cleaning: 64
'AAMI Stadium'
'AAMI Stadium\n'
'Accor Stadium'
'Accor Stadium\n'
'Adelaide Hills'
'Adelaide Oval'
'Adelaide Oval\n'
'Aegis Park'
'Aegis Park\n'
'Barossa Park'


**Issue found in `venue`:** trailing newline characters (\n`) on roughly half the venue
values, which silently doubles up venues like `"Marvel Stadium"` vs `"Marvel Stadium\n"`.


In [15]:
print('missing crowd values:', tm['crowd'].isna().sum())
print('round values:', sorted(tm['round'].unique(), key=lambda x: (len(x), x)))


missing crowd values: 398
round values: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', 'EF', 'GF', 'PF', 'QF', 'SF']


## 5. Apply the cleaning

Strip whitespace, fix casing, standardize the Bulldogs name, and strip venue newlines.

In [16]:
tm_clean = tm.copy()

tm_clean['team_name'] = (
    tm_clean['team_name'].str.strip().str.title().replace({'W. Bulldogs': 'Western Bulldogs'})
)
tm_clean['opponent'] = (
    tm_clean['opponent'].str.strip().str.title().replace({'W. Bulldogs': 'Western Bulldogs'})
)
tm_clean['venue'] = tm_clean['venue'].str.strip()
tm_clean['home_away'] = tm_clean['home_away'].str.strip()

print('team_name unique after cleaning:', tm_clean['team_name'].nunique())
print('opponent unique after cleaning:', tm_clean['opponent'].nunique())
print('venue unique after cleaning:', tm_clean['venue'].nunique())

assert set(tm_clean['team_name'].unique()) == set(raw['team'].unique())
assert set(tm_clean['opponent'].unique()) == set(raw['opponent'].unique())
print('team/opponent names now match the player stats file exactly')


team_name unique after cleaning: 20
opponent unique after cleaning: 20
venue unique after cleaning: 37
team/opponent names now match the player stats file exactly


## 6. Deduplicate the raw player stats file

In [17]:
raw_dedup = raw.drop_duplicates()
print('raw rows before dedup:', len(raw))
print('raw rows after dedup:', len(raw_dedup))


raw rows before dedup: 274089
raw rows after dedup: 274079


## 7. Save cleaned files for the merge notebook

In [18]:
raw_dedup.to_csv('player_stats_cleaned_for_merge.csv', index=False)
tm_clean.to_csv('team_matches_cleaned_for_merge.csv', index=False)
print('saved player_stats_cleaned_for_merge.csv and team_matches_cleaned_for_merge.csv')


saved player_stats_cleaned_for_merge.csv and team_matches_cleaned_for_merge.csv
